<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/10_threshold_diagnostics_FULL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB10_FULL — Diagnóstico metodológico de limiar, granularidade, K/H e persistência mínima por célula

## 1. Contexto

Este notebook inicia a etapa pós-NB06 do ramo `_FULL`, conforme a estratégia v1.5 do pipeline completo. Os notebooks NB00–NB09 pertencem ao Primeiro Ciclo DSR e permanecem como linha histórica congelada; portanto, não existem NB07_FULL, NB08_FULL ou NB09_FULL. No ramo `_FULL`, a seleção de cenários nasce neste NB10_FULL, a modelagem nasce no NB11_FULL e a consolidação ocorre no NB14_FULL–NB16_FULL.

A unidade experimental é a célula Borg (`a`–`h`). Cada célula é processada como réplica independente, sem concatenação temporal das oito células como se fossem uma única série. As saídas são gravadas exclusivamente em:

```text
/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics
```

com subpastas `cell_<id>` e `aggregate`.

## 2. Entradas

Entradas oficiais esperadas:

1. `04-reports/99_FULL_downstream/04_FULL_episodes/cell_<id>/04_FULL_window_5min_series_scored_TRAIN_M2S_cell_<id>.parquet`
2. `04-reports/99_FULL_downstream/04_FULL_episodes/cell_<id>/04_FULL_detect_episodes_summary_cell_<id>.json`
3. `04-reports/99_FULL_downstream/04_FULL_episodes/aggregate/04_FULL_modelability_gate_by_cell.csv`
4. `04-reports/99_FULL_downstream/06_FULL_labeling_states/reports/06_FULL_labeling_states_summary_cell_<id>.json` como fonte auxiliar de rastreabilidade.

A pasta oficial do NB06 é `06_FULL_labeling_states`. Qualquer pasta residual `06_FULL_labeling` é ignorada por este notebook.

## 3. Travas metodológicas

O corte treino/teste não é rederivado por fração local. O NB10_FULL herda, por célula, `train_cutoff_idx` e `train_cutoff_bucket_id` do upstream NB04/NB06. Se esses campos não estiverem presentes nos artefatos anteriores, a execução falha com erro explícito.

O corte 80/20 é preservado como propriedade herdada do NB04_FULL; a validação local apenas verifica que a razão `train_cutoff_idx / n_janelas` permanece próxima de 0,80, sem recalcular o ponto de corte por fração local.

## 4. Cenários avaliados

São preservados os cenários metodológicos do NB10 canônico:

| Grupo | Janela | K | H | Papel |
|---|---:|---:|---:|---|
| `W5_K24_H12` | 5 min | 24 | 12 | ponte principal com o canônico |
| `W5_K12_H6` | 5 min | 12 | 6 | cenário curto 60/30 |
| `W5_K24_H6` | 5 min | 24 | 6 | isola efeito de H |
| `W10_K12_H6` | 10 min | 12 | 6 | versão suavizada 120/60 |
| `W10_K6_H3` | 10 min | 6 | 3 | versão suavizada 60/30 |
| `W10_K12_H3` | 10 min | 12 | 3 | isola H em 10 min |

Os limiares avaliados são `global_m2s`, `train_m2s`, `train_p95` e `train_p99`. O `global_m2s` é mantido apenas como referência histórica retrospectiva e nunca avança como cenário causal para o NB11_FULL.

## 5. Diferenças em relação ao NB10 canônico

1. Entrada: artefatos `_FULL` por célula, não o diretório canônico `03-features`.
2. Execução: iteração sobre `ACTIVE_CELLS = list("abcdefgh")`.
3. Split: corte herdado do NB04/NB06 por `train_cutoff_idx`, nunca rederivado por fração local.
4. Limiar: estimado por célula e por cenário.
5. Saída: `99_FULL_downstream/10_FULL_threshold_diagnostics/cell_<id>` e `aggregate`.
6. Auditoria: `10_FULL_delta_vs_canonical.csv`, diff best-effort e manifesto SHA-256.
7. Lógica científica: mantida — episódios, estados, alvo supervisionado, K/H, persistência e métricas de impacto seguem a formulação do NB10 canônico.


In [4]:
# ============================================================
# NB10_FULL — Diagnóstico metodológico por célula
# Pipeline PPCOMP_DM / ramo _FULL
# ============================================================

from pathlib import Path
import os
import sys
import json
import hashlib
import random
import subprocess
import importlib
import warnings
import difflib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def log(msg: str) -> None:
    print(f"[NB10_FULL_threshold_diagnostics] {msg}")


# ─────────────────────────────────────────────────────────────
# BLOCO 0 — Bootstrap e caminhos oficiais _FULL
# ─────────────────────────────────────────────────────────────

if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível montar o Google Drive automaticamente.")
        print("[Bootstrap] Detalhe:", e)
else:
    print("[Bootstrap] Google Drive já montado.")


MYDRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_ROOT = MYDRIVE_ROOT / "Mestrado"
REPO_DIR = DRIVE_ROOT / "PPCOMP_DM"
NOTEBOOKS_DIR = MYDRIVE_ROOT / "Colab Notebooks"
CANONICAL_NOTEBOOK = NOTEBOOKS_DIR / "10_threshold_diagnostics.ipynb"
FULL_NOTEBOOK = NOTEBOOKS_DIR / "10_threshold_diagnostics_FULL.ipynb"
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull)...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
importlib.invalidate_caches()

FULL_CELLS = list("abcdefgh")
ACTIVE_CELLS = FULL_CELLS[:]  # Para piloto: ACTIVE_CELLS = ["a"]
# ACTIVE_CELLS = ["a"]

FULL_MODEL_FACING_PARQUET = (
    DRIVE_ROOT
    / "02-datasets"
    / "99-full"
    / "03-model-facing"
    / "window_5min_series_allcells_model_facing_000000000000.parquet"
)
CANONICAL_WINDOW_5MIN_SERIES = DRIVE_ROOT / "02-datasets" / "03-features" / "window_5min_series.parquet"

FULL_REPORTS_DIR = DRIVE_ROOT / "04-reports" / "99_FULL_downstream"
STAGE04_DIR = FULL_REPORTS_DIR / "04_FULL_episodes"
STAGE05_DIR = FULL_REPORTS_DIR / "05_FULL_features"
STAGE06_DIR = FULL_REPORTS_DIR / "06_FULL_labeling_states"  # pasta oficial v1.5
LEGACY_STAGE06_DIR = FULL_REPORTS_DIR / "06_FULL_labeling"   # pasta residual: não usar
STAGE10_DIR = FULL_REPORTS_DIR / "10_FULL_threshold_diagnostics"
AGGREGATE_DIR = STAGE10_DIR / "aggregate"
FIGURES_BY_CELL_DIR = AGGREGATE_DIR / "figures_by_cell"

for p in [FULL_REPORTS_DIR, STAGE10_DIR, AGGREGATE_DIR, FIGURES_BY_CELL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT =", DRIVE_ROOT)
print("FULL_REPORTS_DIR =", FULL_REPORTS_DIR)
print("STAGE04_DIR =", STAGE04_DIR)
print("STAGE06_DIR =", STAGE06_DIR)
print("STAGE10_DIR =", STAGE10_DIR)
print("AGGREGATE_DIR =", AGGREGATE_DIR)
print("ACTIVE_CELLS =", ACTIVE_CELLS)

assert "03-features" not in str(FULL_REPORTS_DIR), "FULL_REPORTS_DIR não pode apontar para 03-features."
assert FULL_MODEL_FACING_PARQUET != CANONICAL_WINDOW_5MIN_SERIES, "Entrada FULL não pode ser a série canônica."
assert STAGE04_DIR.exists(), f"Artefatos NB04_FULL não encontrados: {STAGE04_DIR}"
assert STAGE06_DIR.exists(), f"Pasta oficial NB06_FULL não encontrada: {STAGE06_DIR}"
if LEGACY_STAGE06_DIR.exists():
    log(f"Aviso: pasta residual ignorada conforme estratégia v1.5: {LEGACY_STAGE06_DIR}")

CANONICAL_TEMPORAL_CORE = [
    "fail_rate",
    "n_events",
    "n_failed",
    "n_machines",
    "n_collections",
    "event_FAIL_count",
    "event_LOST_count",
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_1h",
    "rolling_std_1h",
    "pct_change",
    "zscore_expanding",
]
temporal_core = CANONICAL_TEMPORAL_CORE[:]
assert len(temporal_core) == 14
assert temporal_core == CANONICAL_TEMPORAL_CORE

# Trava leve sobre a entrada FULL selada. Não é a entrada principal do NB10,
# mas confirma que o mundo _FULL possui as oito células esperadas.
if FULL_MODEL_FACING_PARQUET.exists():
    df_cells_check = pd.read_parquet(FULL_MODEL_FACING_PARQUET, columns=["cell_id"])
    observed_cells = sorted(df_cells_check["cell_id"].dropna().astype(str).unique().tolist())
    assert observed_cells == FULL_CELLS, f"cell_id inesperado no Parquet FULL: {observed_cells}"
else:
    log(f"Aviso: Parquet model-facing FULL não encontrado para checagem leve: {FULL_MODEL_FACING_PARQUET}")


# ─────────────────────────────────────────────────────────────
# BLOCO 1 — Parâmetros herdados do NB10 canônico
# ─────────────────────────────────────────────────────────────

TRAIN_FRACTION_REFERENCE = 0.80  # apenas validação de herança; não rederivar split por fração
STD_DDOF = 0
BASE_WINDOW_MINUTES = 5

PERSISTENCE_VALUES = [1, 2, 3]
THRESHOLD_METHODS = ["global_m2s", "train_m2s", "train_p95", "train_p99"]

TEMPORAL_SCENARIOS = [
    {
        "scenario_family": "W5_K24_H12",
        "window_minutes": 5,
        "k_before_after": 24,
        "horizon_h": 12,
        "role": "historical_bridge_120_60",
        "description": "Janela 5min, BEFORE/AFTER 120min, horizonte 60min; ponte direta com o canônico.",
    },
    {
        "scenario_family": "W5_K12_H6",
        "window_minutes": 5,
        "k_before_after": 12,
        "horizon_h": 6,
        "role": "shorter_operational_60_30",
        "description": "Janela 5min, BEFORE/AFTER 60min, horizonte 30min.",
    },
    {
        "scenario_family": "W5_K24_H6",
        "window_minutes": 5,
        "k_before_after": 24,
        "horizon_h": 6,
        "role": "isolate_horizon_effect_120_30",
        "description": "Janela 5min, BEFORE/AFTER 120min, horizonte 30min; isola redução de H.",
    },
    {
        "scenario_family": "W10_K12_H6",
        "window_minutes": 10,
        "k_before_after": 12,
        "horizon_h": 6,
        "role": "smoothed_120_60",
        "description": "Janela 10min, BEFORE/AFTER 120min, horizonte 60min.",
    },
    {
        "scenario_family": "W10_K6_H3",
        "window_minutes": 10,
        "k_before_after": 6,
        "horizon_h": 3,
        "role": "smoothed_shorter_60_30",
        "description": "Janela 10min, BEFORE/AFTER 60min, horizonte 30min.",
    },
    {
        "scenario_family": "W10_K12_H3",
        "window_minutes": 10,
        "k_before_after": 12,
        "horizon_h": 3,
        "role": "smoothed_isolate_horizon_120_30",
        "description": "Janela 10min, BEFORE/AFTER 120min, horizonte 30min; isola redução de H.",
    },
]

MIN_EPISODES_ADVANCE = 50
MIN_POSITIVE_RATE_ADVANCE = 0.03
MAX_POSITIVE_RATE_ADVANCE = 0.50
MIN_SUPERVISED_ROWS_ADVANCE = 500
MIN_POSITIVES_TRAIN_ADVANCE = 10
MIN_POSITIVES_TEST_ADVANCE = 10

log("Parâmetros principais definidos.")
print("TRAIN_FRACTION_REFERENCE =", TRAIN_FRACTION_REFERENCE, "(somente assert; split herdado do NB04/NB06)")
print("STD_DDOF =", STD_DDOF)
print("PERSISTENCE_VALUES =", PERSISTENCE_VALUES)
print("THRESHOLD_METHODS =", THRESHOLD_METHODS)
print("TEMPORAL_SCENARIOS =", len(TEMPORAL_SCENARIOS))


# ─────────────────────────────────────────────────────────────
# BLOCO 2 — Funções auxiliares gerais
# ─────────────────────────────────────────────────────────────


def as_float(x):
    if pd.isna(x):
        return None
    return float(x)


def as_int(x):
    if pd.isna(x):
        return None
    return int(x)


def format_float(x, digits=6):
    if x is None or pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"


def json_ready(obj):
    """Converte objetos NumPy/Pandas/Path para JSON seguro."""
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return None if np.isnan(val) else val
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def recursive_find_key(obj, key: str):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for value in obj.values():
            found = recursive_find_key(value, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = recursive_find_key(value, key)
            if found is not None:
                return found
    return None


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def build_artifact_manifest(root_dir: Path, out_file: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(root_dir.rglob("*")):
        if not p.is_file():
            continue
        if p == out_file:
            continue
        rows.append({
            "relative_path": str(p.relative_to(root_dir)).replace("\\", "/"),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_file, index=False, encoding="utf-8")
    return df


def ensure_numeric_columns(df: pd.DataFrame, columns, fill_value=0.0) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = fill_value
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(fill_value)
    return out


def normalize_modelability_status(value) -> str:
    if value is None or pd.isna(value):
        return "unknown"
    s = str(value).strip().lower()
    if "descr" in s:
        return "descriptive"
    if "model" in s or "pred" in s or "supervision" in s:
        return "modelable"
    if s in {"ok", "true", "1", "yes", "sim"}:
        return "modelable"
    if s in {"false", "0", "no", "não", "nao"}:
        return "descriptive"
    return s


# ─────────────────────────────────────────────────────────────
# BLOCO 3 — Resolução de entradas e corte herdado
# ─────────────────────────────────────────────────────────────


def load_modelability_gate() -> pd.DataFrame:
    path = STAGE04_DIR / "aggregate" / "04_FULL_modelability_gate_by_cell.csv"
    if not path.exists():
        log(f"Aviso: gate de modelabilidade NB04 não encontrado: {path}")
        return pd.DataFrame({"cell_id": FULL_CELLS, "status_modelagem": "unknown", "justificativa": "gate ausente"})
    df = pd.read_csv(path)
    if "cell_id" not in df.columns:
        raise ValueError(f"Gate de modelabilidade sem cell_id: {path}")
    return df


def get_modelability_for_cell(gate_df: pd.DataFrame, cell_id: str) -> dict:
    rows = gate_df[gate_df["cell_id"].astype(str) == str(cell_id)].copy()
    if rows.empty:
        return {"status_modelagem": "unknown", "status_normalized": "unknown", "justificativa": "cell_id ausente no gate"}
    row = rows.iloc[0].to_dict()
    status_col = "status_modelagem" if "status_modelagem" in rows.columns else None
    if status_col is None:
        for candidate in ["status", "modelability_status", "modelagem", "status_modelability"]:
            if candidate in rows.columns:
                status_col = candidate
                break
    raw_status = row.get(status_col, "unknown") if status_col else "unknown"
    row["status_normalized"] = normalize_modelability_status(raw_status)
    if "justificativa" not in row:
        for c in ["justification", "reason", "motivo"]:
            if c in row:
                row["justificativa"] = row[c]
                break
    row.setdefault("justificativa", "")
    return row


def resolve_nb04_series_file(cell_id: str) -> Path:
    exact = STAGE04_DIR / f"cell_{cell_id}" / f"04_FULL_window_5min_series_scored_TRAIN_M2S_cell_{cell_id}.parquet"
    if exact.exists():
        return exact

    cell_dir = STAGE04_DIR / f"cell_{cell_id}"
    patterns = [
        f"*series*scored*cell_{cell_id}*.parquet",
        f"*window_5min_series*cell_{cell_id}*.parquet",
        f"*scored*TRAIN_M2S*cell_{cell_id}*.parquet",
    ]
    candidates = []
    if cell_dir.exists():
        for pattern in patterns:
            candidates.extend(sorted(cell_dir.glob(pattern)))
    candidates = [p for p in candidates if "03-features" not in str(p)]
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise FileExistsError(f"Mais de um candidato NB04 para cell_{cell_id}: {[str(p) for p in candidates]}")
    raise FileNotFoundError(f"Série scored do NB04_FULL não encontrada para cell_{cell_id}: {exact}")


def resolve_summary_jsons_for_cell(cell_id: str):
    paths = [
        STAGE04_DIR / f"cell_{cell_id}" / f"04_FULL_detect_episodes_summary_cell_{cell_id}.json",
        STAGE06_DIR / "reports" / f"06_FULL_labeling_states_summary_cell_{cell_id}.json",
    ]
    return [p for p in paths if p.exists()]


def get_inherited_cutoff(df_series: pd.DataFrame, cell_id: str) -> dict:
    cutoff_idx = None
    cutoff_bucket = None
    source = None

    if {"train_cutoff_idx", "train_cutoff_bucket_id"}.issubset(df_series.columns):
        cutoff_idx = df_series["train_cutoff_idx"].dropna().iloc[0] if df_series["train_cutoff_idx"].notna().any() else None
        cutoff_bucket = df_series["train_cutoff_bucket_id"].dropna().iloc[0] if df_series["train_cutoff_bucket_id"].notna().any() else None
        source = "NB04_series_columns"

    if cutoff_idx is None or cutoff_bucket is None:
        for path in resolve_summary_jsons_for_cell(cell_id):
            payload = read_json(path)
            idx_candidate = recursive_find_key(payload, "train_cutoff_idx")
            bucket_candidate = recursive_find_key(payload, "train_cutoff_bucket_id")
            if idx_candidate is not None and bucket_candidate is not None:
                cutoff_idx = idx_candidate
                cutoff_bucket = bucket_candidate
                source = str(path)
                break

    assert cutoff_idx is not None and cutoff_bucket is not None, (
        f"NB04/NB06 sem corte herdado para cell_{cell_id}: "
        "train_cutoff_idx e train_cutoff_bucket_id são obrigatórios; abortar, não recalcular por fração."
    )

    cutoff_idx = int(cutoff_idx)
    cutoff_bucket = int(cutoff_bucket)
    return {
        "train_cutoff_idx": cutoff_idx,
        "train_cutoff_bucket_id": cutoff_bucket,
        "train_cutoff_source": source,
    }


def validate_cutoff_for_series(df: pd.DataFrame, cutoff_idx: int, cell_id: str, window_minutes: int) -> None:
    if cutoff_idx <= 0 or cutoff_idx >= len(df):
        raise ValueError(f"Corte herdado inválido para cell_{cell_id} W{window_minutes}: cutoff_idx={cutoff_idx}, len={len(df)}")
    ratio = cutoff_idx / len(df)
    assert abs(ratio - TRAIN_FRACTION_REFERENCE) < 0.01, (
        f"Corte herdado não preserva 80/20 para cell_{cell_id} W{window_minutes}: "
        f"cutoff_idx={cutoff_idx}, len={len(df)}, ratio={ratio:.4f}"
    )


def prepare_5min_series(df_raw: pd.DataFrame, cell_id: str, cutoff: dict) -> pd.DataFrame:
    df = df_raw.copy()
    assert "cell_id" in df.columns, "Série NB04_FULL deve conter cell_id."
    cells = sorted(df["cell_id"].dropna().astype(str).unique().tolist())
    assert cells == [cell_id], f"Série da célula {cell_id} contém cell_id inesperado: {cells}"
    assert "bucket_id" in df.columns, "Série NB04_FULL deve conter bucket_id."

    df = df.sort_values("bucket_id").drop_duplicates(subset=["bucket_id"], keep="first").reset_index(drop=True)
    df["bucket_id"] = df["bucket_id"].astype(int)
    if "bucket_start_us" not in df.columns:
        df["bucket_start_us"] = df["bucket_id"] * BASE_WINDOW_MINUTES * 60 * 1_000_000

    raw_cols = [
        "n_events", "n_failed", "n_machines", "n_collections",
        "event_FAIL_count", "event_SCHEDULE_count", "event_FINISH_count",
        "event_ENABLE_count", "event_LOST_count", "event_EVICT_count", "event_KILL_count",
    ]
    df = ensure_numeric_columns(df, raw_cols, fill_value=0.0)
    df["fail_rate"] = np.where(df["n_events"] > 0, df["n_failed"] / df["n_events"], 0.0)
    df["cell_id"] = cell_id
    df["window_minutes"] = 5
    df["row_position"] = np.arange(len(df), dtype=int)

    cutoff_idx = int(cutoff["train_cutoff_idx"])
    validate_cutoff_for_series(df, cutoff_idx, cell_id, 5)
    df["train_cutoff_idx"] = cutoff_idx
    df["train_cutoff_bucket_id"] = int(cutoff["train_cutoff_bucket_id"])
    df["train_cutoff_source"] = cutoff["train_cutoff_source"]
    df["temporal_partition"] = np.where(df["row_position"] < cutoff_idx, "train", "test")
    assert df.loc[df["temporal_partition"] == "train", "row_position"].max() + 1 == cutoff_idx
    return df


def prepare_10min_series(df_5: pd.DataFrame, cell_id: str, cutoff: dict) -> pd.DataFrame:
    df = df_5.copy()
    df["bucket_10_id"] = (df["bucket_id"] // 2).astype(int)
    sum_cols = [
        "n_events", "n_failed", "n_machines", "n_collections",
        "event_FAIL_count", "event_SCHEDULE_count", "event_FINISH_count",
        "event_ENABLE_count", "event_LOST_count", "event_EVICT_count", "event_KILL_count",
    ]
    agg_dict = {"bucket_start_us": "min", "bucket_id": ["min", "max"]}
    for col in sum_cols:
        agg_dict[col] = "sum"

    out = df.groupby("bucket_10_id", as_index=False).agg(agg_dict)
    flat_cols = []
    for col in out.columns:
        if isinstance(col, tuple):
            base, stat = col
            if base == "bucket_10_id":
                flat_cols.append("bucket_10_id")
            elif base == "bucket_id" and stat == "min":
                flat_cols.append("first_bucket_5min")
            elif base == "bucket_id" and stat == "max":
                flat_cols.append("last_bucket_5min")
            else:
                flat_cols.append(str(base))
        else:
            flat_cols.append(str(col))
    out.columns = flat_cols
    out = out.rename(columns={"bucket_10_id": "bucket_id"}).sort_values("bucket_id").reset_index(drop=True)

    out["fail_rate"] = np.where(out["n_events"] > 0, out["n_failed"] / out["n_events"], 0.0)
    out["cell_id"] = cell_id
    out["window_minutes"] = 10
    out["row_position"] = np.arange(len(out), dtype=int)

    cutoff_10_idx = int(cutoff["train_cutoff_idx"]) // 2
    cutoff_10_bucket = int(cutoff["train_cutoff_bucket_id"]) // 2
    validate_cutoff_for_series(out, cutoff_10_idx, cell_id, 10)
    out["train_cutoff_idx"] = cutoff_10_idx
    out["train_cutoff_bucket_id"] = cutoff_10_bucket
    out["train_cutoff_source"] = "derived_from_5min_cutoff_pos_div_2"
    out["temporal_partition"] = np.where(out["row_position"] < cutoff_10_idx, "train", "test")
    assert out.loc[out["temporal_partition"] == "train", "row_position"].max() + 1 == cutoff_10_idx
    return out


# ─────────────────────────────────────────────────────────────
# BLOCO 4 — Funções científicas do NB10
# ─────────────────────────────────────────────────────────────


def descriptive_stats(values: pd.Series, scope: str, window_minutes: int, cell_id: str) -> dict:
    values = values.dropna().astype(float)
    return {
        "cell_id": cell_id,
        "window_minutes": int(window_minutes),
        "scope": scope,
        "count": int(values.shape[0]),
        "zero_count": int((values == 0).sum()),
        "zero_rate": float((values == 0).mean()) if len(values) else np.nan,
        "nonzero_count": int((values > 0).sum()),
        "mean": float(values.mean()) if len(values) else np.nan,
        "std": float(values.std(ddof=STD_DDOF)) if len(values) else np.nan,
        "min": float(values.min()) if len(values) else np.nan,
        "p01": float(values.quantile(0.01)) if len(values) else np.nan,
        "p05": float(values.quantile(0.05)) if len(values) else np.nan,
        "p25": float(values.quantile(0.25)) if len(values) else np.nan,
        "median": float(values.quantile(0.50)) if len(values) else np.nan,
        "p75": float(values.quantile(0.75)) if len(values) else np.nan,
        "p90": float(values.quantile(0.90)) if len(values) else np.nan,
        "p95": float(values.quantile(0.95)) if len(values) else np.nan,
        "p99": float(values.quantile(0.99)) if len(values) else np.nan,
        "max": float(values.max()) if len(values) else np.nan,
    }


def compute_thresholds(df: pd.DataFrame, window_minutes: int, cell_id: str) -> pd.DataFrame:
    full_fail = df["fail_rate"].astype(float)
    train_fail = df.loc[df["temporal_partition"] == "train", "fail_rate"].astype(float)
    if train_fail.empty:
        raise ValueError(f"Trecho de treino vazio para cell_{cell_id} W{window_minutes}.")

    global_mu = float(full_fail.mean())
    global_std = float(full_fail.std(ddof=STD_DDOF))
    global_p95 = float(full_fail.quantile(0.95))
    global_p99 = float(full_fail.quantile(0.99))
    train_mu = float(train_fail.mean())
    train_std = float(train_fail.std(ddof=STD_DDOF))
    train_p95 = float(train_fail.quantile(0.95))
    train_p99 = float(train_fail.quantile(0.99))

    rows = [
        {
            "cell_id": cell_id,
            "window_minutes": int(window_minutes),
            "threshold_method": "global_m2s",
            "description": "Limiar global retrospectivo μ + 2σ calculado sobre toda a série; referência histórica, não causal",
            "fit_scope": "full_series",
            "uses_test_statistics": True,
            "mu": global_mu,
            "std": global_std,
            "p95": global_p95,
            "p99": global_p99,
            "threshold_value": global_mu + 2 * global_std,
            "train_cutoff_idx": int(df["train_cutoff_idx"].iloc[0]),
            "train_cutoff_bucket_id": int(df["train_cutoff_bucket_id"].iloc[0]),
        },
        {
            "cell_id": cell_id,
            "window_minutes": int(window_minutes),
            "threshold_method": "train_m2s",
            "description": "Limiar causal μ + 2σ estimado apenas no treino herdado do NB04/NB06",
            "fit_scope": "train_only",
            "uses_test_statistics": False,
            "mu": train_mu,
            "std": train_std,
            "p95": train_p95,
            "p99": train_p99,
            "threshold_value": train_mu + 2 * train_std,
            "train_cutoff_idx": int(df["train_cutoff_idx"].iloc[0]),
            "train_cutoff_bucket_id": int(df["train_cutoff_bucket_id"].iloc[0]),
        },
        {
            "cell_id": cell_id,
            "window_minutes": int(window_minutes),
            "threshold_method": "train_p95",
            "description": "Limiar causal p95 estimado apenas no treino herdado do NB04/NB06",
            "fit_scope": "train_only",
            "uses_test_statistics": False,
            "mu": train_mu,
            "std": train_std,
            "p95": train_p95,
            "p99": train_p99,
            "threshold_value": train_p95,
            "train_cutoff_idx": int(df["train_cutoff_idx"].iloc[0]),
            "train_cutoff_bucket_id": int(df["train_cutoff_bucket_id"].iloc[0]),
        },
        {
            "cell_id": cell_id,
            "window_minutes": int(window_minutes),
            "threshold_method": "train_p99",
            "description": "Limiar causal p99 estimado apenas no treino; diagnóstico de severidade extrema",
            "fit_scope": "train_only",
            "uses_test_statistics": False,
            "mu": train_mu,
            "std": train_std,
            "p95": train_p95,
            "p99": train_p99,
            "threshold_value": train_p99,
            "train_cutoff_idx": int(df["train_cutoff_idx"].iloc[0]),
            "train_cutoff_bucket_id": int(df["train_cutoff_bucket_id"].iloc[0]),
        },
    ]
    return pd.DataFrame(rows)


def detect_episodes_from_flag(df: pd.DataFrame, flag_col: str) -> pd.DataFrame:
    flags = df[flag_col].fillna(False).astype(bool).to_numpy()
    episodes = []
    in_episode = False
    start_pos = None

    for i, is_critical in enumerate(flags):
        if is_critical and not in_episode:
            in_episode = True
            start_pos = i
        if in_episode and ((not is_critical) or i == len(flags) - 1):
            end_pos = i if is_critical and i == len(flags) - 1 else i - 1
            segment = df.iloc[start_pos:end_pos + 1]
            episodes.append({
                "episode_id": len(episodes),
                "start_pos": int(start_pos),
                "end_pos": int(end_pos),
                "start_bucket_id": int(segment["bucket_id"].iloc[0]),
                "end_bucket_id": int(segment["bucket_id"].iloc[-1]),
                "start_bucket_start_us": int(segment["bucket_start_us"].iloc[0]),
                "end_bucket_start_us": int(segment["bucket_start_us"].iloc[-1]),
                "duration_windows": int(end_pos - start_pos + 1),
                "max_fail_rate": float(segment["fail_rate"].max()),
                "mean_fail_rate": float(segment["fail_rate"].mean()),
                "sum_n_events": float(segment["n_events"].sum()) if "n_events" in segment.columns else np.nan,
                "sum_n_failed": float(segment["n_failed"].sum()) if "n_failed" in segment.columns else np.nan,
            })
            in_episode = False
            start_pos = None
    return pd.DataFrame(episodes)


def build_valid_critical_flag(df: pd.DataFrame, threshold_value: float, persistence_min_windows: int) -> pd.DataFrame:
    out = df.copy()
    out["is_critical_raw"] = out["fail_rate"].astype(float) >= float(threshold_value)
    raw_episodes = detect_episodes_from_flag(out, "is_critical_raw")
    out["is_critical"] = False
    valid_rows = []
    for _, ep in raw_episodes.iterrows():
        is_valid = int(ep["duration_windows"]) >= int(persistence_min_windows)
        ep_dict = ep.to_dict()
        ep_dict["raw_episode_id"] = int(ep_dict.pop("episode_id"))
        ep_dict["persistence_min_windows"] = int(persistence_min_windows)
        ep_dict["valid_after_persistence"] = bool(is_valid)
        valid_rows.append(ep_dict)
        if is_valid:
            out.loc[int(ep["start_pos"]):int(ep["end_pos"]), "is_critical"] = True
    valid_eps = pd.DataFrame(valid_rows)
    episodes = detect_episodes_from_flag(out, "is_critical")
    return out, raw_episodes, valid_eps, episodes


def assign_states_from_episodes(df: pd.DataFrame, episodes: pd.DataFrame, k_before_after: int) -> pd.DataFrame:
    out = df.copy()
    n = len(out)
    out["state"] = "NORMAL"

    # AFTER primeiro, BEFORE depois, DURING por último: preserva precedência DURING > BEFORE > AFTER > NORMAL.
    for _, ep in episodes.iterrows():
        start = int(ep["start_pos"])
        end = int(ep["end_pos"])
        after_start = end + 1
        after_end = min(n - 1, end + int(k_before_after))
        if after_start <= after_end:
            mask = out.index.to_series().between(after_start, after_end)
            out.loc[mask & (out["state"] == "NORMAL"), "state"] = "AFTER"

    for _, ep in episodes.iterrows():
        start = int(ep["start_pos"])
        before_start = max(0, start - int(k_before_after))
        before_end = start - 1
        if before_start <= before_end:
            mask = out.index.to_series().between(before_start, before_end)
            out.loc[mask & (out["state"] != "DURING"), "state"] = "BEFORE"

    for _, ep in episodes.iterrows():
        start = int(ep["start_pos"])
        end = int(ep["end_pos"])
        out.loc[start:end, "state"] = "DURING"

    return out


def build_supervised_target(df_states: pd.DataFrame, horizon_h: int) -> pd.DataFrame:
    out = df_states.copy()
    state = out["state"].astype(str).to_numpy()
    n = len(out)
    y = np.full(n, np.nan)
    future_complete = np.zeros(n, dtype=bool)

    for i in range(n):
        if state[i] not in {"NORMAL", "BEFORE"}:
            continue
        if i + int(horizon_h) >= n:
            continue
        future_complete[i] = True
        future_window = state[i + 1:i + int(horizon_h) + 1]
        y[i] = 1 if np.any(future_window == "DURING") else 0

    out["horizon_h"] = int(horizon_h)
    out["future_complete"] = future_complete
    out["y_anticipation"] = y
    return out


def summarize_episode_scenario(episodes: pd.DataFrame, scenario_meta: dict) -> dict:
    row = dict(scenario_meta)
    row["episode_count"] = int(len(episodes))
    if len(episodes) == 0:
        row.update({
            "total_critical_windows": 0,
            "mean_duration_windows": np.nan,
            "median_duration_windows": np.nan,
            "max_duration_windows": np.nan,
            "one_window_episodes": 0,
            "mean_max_fail_rate": np.nan,
            "max_fail_rate_overall": np.nan,
        })
        return row
    row.update({
        "total_critical_windows": int(episodes["duration_windows"].sum()),
        "mean_duration_windows": float(episodes["duration_windows"].mean()),
        "median_duration_windows": float(episodes["duration_windows"].median()),
        "max_duration_windows": int(episodes["duration_windows"].max()),
        "one_window_episodes": int((episodes["duration_windows"] == 1).sum()),
        "mean_max_fail_rate": float(episodes["max_fail_rate"].mean()),
        "max_fail_rate_overall": float(episodes["max_fail_rate"].max()),
    })
    return row


def summarize_label_impact(df_target: pd.DataFrame, scenario_meta: dict) -> dict:
    valid = df_target[df_target["y_anticipation"].notna()].copy()
    row = dict(scenario_meta)
    row["supervised_rows"] = int(len(valid))
    if len(valid) == 0:
        row.update({
            "positives": 0,
            "negatives": 0,
            "positive_rate": np.nan,
            "n_positivos_train": 0,
            "n_positivos_test": 0,
            "supervised_rows_train": 0,
            "supervised_rows_test": 0,
            "state_NORMAL_rows": 0,
            "state_BEFORE_rows": 0,
        })
        return row
    valid["y_anticipation"] = valid["y_anticipation"].astype(int)
    train_valid = valid[valid["temporal_partition"] == "train"]
    test_valid = valid[valid["temporal_partition"] == "test"]
    row.update({
        "positives": int(valid["y_anticipation"].sum()),
        "negatives": int((valid["y_anticipation"] == 0).sum()),
        "positive_rate": float(valid["y_anticipation"].mean()),
        "n_positivos_train": int(train_valid["y_anticipation"].sum()) if len(train_valid) else 0,
        "n_positivos_test": int(test_valid["y_anticipation"].sum()) if len(test_valid) else 0,
        "supervised_rows_train": int(len(train_valid)),
        "supervised_rows_test": int(len(test_valid)),
        "state_NORMAL_rows": int((valid["state"] == "NORMAL").sum()),
        "state_BEFORE_rows": int((valid["state"] == "BEFORE").sum()),
    })
    return row


def decide_scenario(row: pd.Series, modelability_status: str) -> dict:
    method = row["threshold_method"]
    persistence = int(row["persistence_min_windows"])
    episode_count = int(row.get("episode_count", 0) or 0)
    supervised_rows = int(row.get("supervised_rows", 0) or 0)
    positive_rate = row.get("positive_rate", np.nan)
    n_pos_train = int(row.get("n_positivos_train", 0) or 0)
    n_pos_test = int(row.get("n_positivos_test", 0) or 0)

    if modelability_status == "descriptive":
        return {
            "decision_category": "do_not_advance",
            "advance_to_nb11": False,
            "scientific_justification": "Célula marcada como descritiva no gate de modelabilidade NB04; não forçar modelagem supervisionada.",
        }

    if method == "global_m2s":
        return {
            "decision_category": "historical_reference",
            "advance_to_nb11": False,
            "scientific_justification": "Limiar global retrospectivo mantido apenas como referência histórica; não avança no ramo causal _FULL.",
        }

    if method == "train_p99":
        return {
            "decision_category": "sensitivity_only",
            "advance_to_nb11": False,
            "scientific_justification": "p99 é diagnóstico de severidade extrema; não é candidato principal para NB11_FULL.",
        }

    if pd.isna(positive_rate):
        viable = False
    else:
        viable = (
            episode_count >= MIN_EPISODES_ADVANCE
            and supervised_rows >= MIN_SUPERVISED_ROWS_ADVANCE
            and MIN_POSITIVE_RATE_ADVANCE <= float(positive_rate) <= MAX_POSITIVE_RATE_ADVANCE
            and n_pos_train >= MIN_POSITIVES_TRAIN_ADVANCE
            and n_pos_test >= MIN_POSITIVES_TEST_ADVANCE
        )

    if method == "train_m2s" and persistence in {1, 2} and viable:
        return {
            "decision_category": "advance_to_nb11",
            "advance_to_nb11": True,
            "scientific_justification": "Cenário causal train_m2s com persistência foco e suporte mínimo em treino/teste; avança para NB11_FULL.",
        }

    if method == "train_p95" and viable:
        return {
            "decision_category": "sensitivity_only",
            "advance_to_nb11": False,
            "scientific_justification": "p95 é sensibilidade causal relevante para assimetria/caudas, mas não substitui o cenário principal train_m2s.",
        }

    if method == "train_m2s" and persistence == 3:
        return {
            "decision_category": "sensitivity_only",
            "advance_to_nb11": False,
            "scientific_justification": "Persistência 3 é restritiva; manter como sensibilidade, não como cenário principal.",
        }

    return {
        "decision_category": "do_not_advance",
        "advance_to_nb11": False,
        "scientific_justification": (
            "Cenário não atende aos critérios mínimos de suporte para avanço "
            f"(episódios={episode_count}, linhas={supervised_rows}, positivos_train={n_pos_train}, positivos_test={n_pos_test}, taxa={format_float(positive_rate, 4)})."
        ),
    }


def save_figure(fig, cell_dir: Path, agg_fig_dir: Path, filename: str) -> dict:
    cell_fig_path = cell_dir / "figures_nb10_scenarios" / filename
    agg_fig_path = agg_fig_dir / filename
    cell_fig_path.parent.mkdir(parents=True, exist_ok=True)
    agg_fig_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(cell_fig_path, dpi=300, bbox_inches="tight")
    fig.savefig(agg_fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return {"cell_path": str(cell_fig_path), "aggregate_copy_path": str(agg_fig_path)}


# ─────────────────────────────────────────────────────────────
# BLOCO 5 — Processamento de uma célula
# ─────────────────────────────────────────────────────────────


def process_cell(cell_id: str, gate_df: pd.DataFrame) -> dict:
    log(f"Iniciando célula {cell_id}")
    cell_dir = STAGE10_DIR / f"cell_{cell_id}"
    cell_dir.mkdir(parents=True, exist_ok=True)
    agg_fig_dir = FIGURES_BY_CELL_DIR / f"cell_{cell_id}"
    agg_fig_dir.mkdir(parents=True, exist_ok=True)

    nb04_series_file = resolve_nb04_series_file(cell_id)
    df_nb04_raw = pd.read_parquet(nb04_series_file)
    cutoff = get_inherited_cutoff(df_nb04_raw, cell_id)
    modelability = get_modelability_for_cell(gate_df, cell_id)
    modelability_status = modelability.get("status_normalized", "unknown")

    df_5 = prepare_5min_series(df_nb04_raw, cell_id, cutoff)
    df_10 = prepare_10min_series(df_5, cell_id, cutoff)

    series_by_window = {5: df_5, 10: df_10}
    figures = {}

    stats_rows = []
    for window_minutes, df in series_by_window.items():
        stats_rows.append(descriptive_stats(df["fail_rate"], "full_series", window_minutes, cell_id))
        stats_rows.append(descriptive_stats(df.loc[df["temporal_partition"] == "train", "fail_rate"], "train", window_minutes, cell_id))
        stats_rows.append(descriptive_stats(df.loc[df["temporal_partition"] == "test", "fail_rate"], "test", window_minutes, cell_id))
    df_stats = pd.DataFrame(stats_rows)

    thresholds_rows = []
    for window_minutes, df in series_by_window.items():
        thresholds_rows.append(compute_thresholds(df, window_minutes, cell_id))
    df_thresholds = pd.concat(thresholds_rows, ignore_index=True)

    # Figuras básicas por granularidade.
    for window_minutes, df in series_by_window.items():
        thr = df_thresholds[df_thresholds["window_minutes"] == window_minutes].copy()
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.hist(df["fail_rate"], bins=40, alpha=0.8)
        for _, row in thr.iterrows():
            ax.axvline(row["threshold_value"], linestyle="--", label=f"{row['threshold_method']}={row['threshold_value']:.4f}")
        ax.set_title(f"Cell {cell_id} — distribuição da fail_rate W{window_minutes}")
        ax.set_xlabel("fail_rate")
        ax.set_ylabel("Frequência")
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(fontsize=8)
        figures[f"w{window_minutes}_fail_rate_hist_thresholds"] = save_figure(
            fig, cell_dir, agg_fig_dir, f"10_FULL_fig_w{window_minutes}_fail_rate_hist_thresholds_cell_{cell_id}.png"
        )

        fig, ax = plt.subplots(figsize=(12, 4.8))
        ax.plot(df["row_position"], df["fail_rate"], linewidth=0.8)
        ax.axvline(int(df["train_cutoff_idx"].iloc[0]), linestyle=":", label="corte treino/teste herdado")
        for _, row in thr.iterrows():
            if row["threshold_method"] in {"train_m2s", "train_p95"}:
                ax.axhline(row["threshold_value"], linestyle="--", label=f"{row['threshold_method']}={row['threshold_value']:.4f}")
        ax.set_title(f"Cell {cell_id} — fail_rate temporal W{window_minutes}")
        ax.set_xlabel("posição temporal")
        ax.set_ylabel("fail_rate")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        figures[f"w{window_minutes}_fail_rate_time_thresholds"] = save_figure(
            fig, cell_dir, agg_fig_dir, f"10_FULL_fig_w{window_minutes}_fail_rate_time_thresholds_cell_{cell_id}.png"
        )

    scenario_episode_rows = []
    scenario_label_rows = []
    scenario_decision_rows = []
    all_episode_frames = []
    all_state_frames = []

    for scenario in TEMPORAL_SCENARIOS:
        window_minutes = int(scenario["window_minutes"])
        df_base = series_by_window[window_minutes]
        thresholds_for_window = df_thresholds[df_thresholds["window_minutes"] == window_minutes].copy()

        for _, thr in thresholds_for_window.iterrows():
            for persistence in PERSISTENCE_VALUES:
                threshold_method = str(thr["threshold_method"])
                scenario_id = (
                    f"cell_{cell_id}_{scenario['scenario_family']}_{threshold_method}_P{persistence}"
                )
                scenario_meta = {
                    "cell_id": cell_id,
                    "scenario_id": scenario_id,
                    "scenario_family": scenario["scenario_family"],
                    "window_minutes": window_minutes,
                    "k_before_after": int(scenario["k_before_after"]),
                    "horizon_h": int(scenario["horizon_h"]),
                    "k_minutes": int(scenario["k_before_after"]) * window_minutes,
                    "h_minutes": int(scenario["horizon_h"]) * window_minutes,
                    "scenario_role": scenario["role"],
                    "threshold_method": threshold_method,
                    "threshold_value": float(thr["threshold_value"]),
                    "persistence_min_windows": int(persistence),
                    "persistence_minutes": int(persistence) * window_minutes,
                    "uses_test_statistics": bool(thr["uses_test_statistics"]),
                    "fit_scope": thr["fit_scope"],
                    "modelability_status": modelability_status,
                    "modelability_justification": modelability.get("justificativa", ""),
                    "train_cutoff_idx": int(df_base["train_cutoff_idx"].iloc[0]),
                    "train_cutoff_bucket_id": int(df_base["train_cutoff_bucket_id"].iloc[0]),
                }

                df_flagged, raw_eps, valid_eps, episodes = build_valid_critical_flag(
                    df_base, float(thr["threshold_value"]), int(persistence)
                )
                df_states = assign_states_from_episodes(df_flagged, episodes, int(scenario["k_before_after"]))
                df_target = build_supervised_target(df_states, int(scenario["horizon_h"]))

                if len(episodes):
                    ep = episodes.copy()
                    for k, v in scenario_meta.items():
                        ep[k] = v
                    all_episode_frames.append(ep)

                state_cols = [
                    "cell_id", "scenario_id", "scenario_family", "window_minutes", "row_position", "bucket_id",
                    "bucket_start_us", "temporal_partition", "fail_rate", "n_events", "n_failed",
                    "is_critical_raw", "is_critical", "state", "future_complete", "y_anticipation",
                ]
                state_frame = df_target.copy()
                for k, v in scenario_meta.items():
                    state_frame[k] = v
                keep_cols = [c for c in state_cols if c in state_frame.columns]
                extra_cols = [
                    "k_before_after", "horizon_h", "threshold_method", "threshold_value",
                    "persistence_min_windows", "modelability_status",
                ]
                keep_cols = keep_cols + [c for c in extra_cols if c in state_frame.columns and c not in keep_cols]
                all_state_frames.append(state_frame[keep_cols])

                ep_summary = summarize_episode_scenario(episodes, scenario_meta)
                label_summary = summarize_label_impact(df_target, scenario_meta)
                scenario_episode_rows.append(ep_summary)
                scenario_label_rows.append(label_summary)

                decision_input = pd.Series({**ep_summary, **label_summary})
                decision = decide_scenario(decision_input, modelability_status)
                scenario_decision_rows.append({**ep_summary, **label_summary, **decision})

    df_episode_summary = pd.DataFrame(scenario_episode_rows)
    df_label_impact = pd.DataFrame(scenario_label_rows)
    df_decisions = pd.DataFrame(scenario_decision_rows)
    df_episodes = pd.concat(all_episode_frames, ignore_index=True) if all_episode_frames else pd.DataFrame()
    df_states = pd.concat(all_state_frames, ignore_index=True) if all_state_frames else pd.DataFrame()
    advanced_scenarios = df_decisions[df_decisions["advance_to_nb11"] == True].copy()

    # Figuras comparativas de cenários principais train_m2s.
    main_plot = df_decisions[
        (df_decisions["threshold_method"] == "train_m2s")
        & (df_decisions["persistence_min_windows"].isin([1, 2, 3]))
    ].copy()
    main_plot["plot_label"] = main_plot["scenario_family"] + "_P" + main_plot["persistence_min_windows"].astype(str)

    if not main_plot.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(main_plot["plot_label"], main_plot["episode_count"])
        ax.set_xlabel("Cenário")
        ax.set_ylabel("Número de episódios")
        ax.set_title(f"Cell {cell_id} — episódios por cenário principal train_m2s")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(True, axis="y", alpha=0.3)
        figures["main_scenarios_episodes"] = save_figure(
            fig, cell_dir, agg_fig_dir, f"10_FULL_fig_main_scenarios_episodes_cell_{cell_id}.png"
        )

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(main_plot["plot_label"], main_plot["positive_rate"].fillna(0))
        ax.set_xlabel("Cenário")
        ax.set_ylabel("Taxa positiva supervisionada")
        ax.set_title(f"Cell {cell_id} — taxa positiva por cenário principal train_m2s")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(True, axis="y", alpha=0.3)
        figures["main_scenarios_positive_rate"] = save_figure(
            fig, cell_dir, agg_fig_dir, f"10_FULL_fig_main_scenarios_positive_rate_cell_{cell_id}.png"
        )

    decision_counts = df_decisions["decision_category"].value_counts().reindex(
        ["advance_to_nb11", "sensitivity_only", "historical_reference", "do_not_advance"], fill_value=0
    )
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.bar(decision_counts.index, decision_counts.values)
    ax.set_xlabel("Categoria de decisão")
    ax.set_ylabel("Quantidade de cenários")
    ax.set_title(f"Cell {cell_id} — classificação metodológica dos cenários")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(True, axis="y", alpha=0.3)
    figures["scenario_decision_categories"] = save_figure(
        fig, cell_dir, agg_fig_dir, f"10_FULL_fig_scenario_decision_categories_cell_{cell_id}.png"
    )

    # Gravação por célula.
    files = {
        "fail_rate_descriptive_stats": cell_dir / f"10_FULL_fail_rate_descriptive_stats_cell_{cell_id}.csv",
        "thresholds_summary": cell_dir / f"10_FULL_thresholds_summary_cell_{cell_id}.csv",
        "episode_summary": cell_dir / f"10_FULL_episode_summary_cell_{cell_id}.csv",
        "label_impact": cell_dir / f"10_FULL_label_impact_cell_{cell_id}.csv",
        "scenario_decisions": cell_dir / f"10_FULL_scenario_decisions_cell_{cell_id}.csv",
        "episodes": cell_dir / f"10_FULL_scenario_episodes_cell_{cell_id}.parquet",
        "scenario_series_states": cell_dir / f"10_FULL_scenario_series_states_cell_{cell_id}.parquet",
        "advanced_scenarios": cell_dir / f"10_FULL_advanced_to_nb11_cell_{cell_id}.json",
        "summary": cell_dir / f"10_FULL_diagnostics_summary_cell_{cell_id}.json",
    }

    df_stats.to_csv(files["fail_rate_descriptive_stats"], index=False, encoding="utf-8")
    df_thresholds.to_csv(files["thresholds_summary"], index=False, encoding="utf-8")
    df_episode_summary.to_csv(files["episode_summary"], index=False, encoding="utf-8")
    df_label_impact.to_csv(files["label_impact"], index=False, encoding="utf-8")
    df_decisions.to_csv(files["scenario_decisions"], index=False, encoding="utf-8")
    df_episodes.to_parquet(files["episodes"], index=False)
    df_states.to_parquet(files["scenario_series_states"], index=False)

    advanced_payload = {
        "notebook": "NB10_FULL_threshold_diagnostics",
        "cell_id": cell_id,
        "modelability": modelability,
        "advanced_to_nb11": advanced_scenarios.to_dict(orient="records"),
        "selection_rules": {
            "min_episodes_advance": MIN_EPISODES_ADVANCE,
            "min_positive_rate_advance": MIN_POSITIVE_RATE_ADVANCE,
            "max_positive_rate_advance": MAX_POSITIVE_RATE_ADVANCE,
            "min_supervised_rows_advance": MIN_SUPERVISED_ROWS_ADVANCE,
            "min_positives_train_advance": MIN_POSITIVES_TRAIN_ADVANCE,
            "min_positives_test_advance": MIN_POSITIVES_TEST_ADVANCE,
            "persistence_focus": [1, 2],
            "main_threshold": "train_m2s",
            "sensitivity_threshold": "train_p95",
            "extreme_threshold": "train_p99",
        },
    }
    files["advanced_scenarios"].write_text(json.dumps(json_ready(advanced_payload), indent=2, ensure_ascii=False), encoding="utf-8")

    summary = {
        "notebook": "NB10_FULL_threshold_diagnostics",
        "cell_id": cell_id,
        "input_nb04_series_file": str(nb04_series_file),
        "stage10_dir": str(cell_dir),
        "parameters": {
            "train_fraction_reference_only": TRAIN_FRACTION_REFERENCE,
            "split_policy": "inherited_from_NB04_NB06_train_cutoff_idx_never_rederived_by_fraction",
            "std_ddof": STD_DDOF,
            "base_window_minutes": BASE_WINDOW_MINUTES,
            "persistence_values": PERSISTENCE_VALUES,
            "threshold_methods": THRESHOLD_METHODS,
            "temporal_scenarios": TEMPORAL_SCENARIOS,
        },
        "cutoff": cutoff,
        "cutoff_validation": {
            "w5_rows": int(len(df_5)),
            "w5_train_cutoff_idx": int(df_5["train_cutoff_idx"].iloc[0]),
            "w5_train_ratio": float(int(df_5["train_cutoff_idx"].iloc[0]) / len(df_5)),
            "w10_rows": int(len(df_10)),
            "w10_train_cutoff_idx": int(df_10["train_cutoff_idx"].iloc[0]),
            "w10_train_ratio": float(int(df_10["train_cutoff_idx"].iloc[0]) / len(df_10)),
        },
        "modelability": modelability,
        "series_shapes": {
            "window_5min_rows": int(len(df_5)),
            "window_10min_rows": int(len(df_10)),
        },
        "decision_counts": df_decisions["decision_category"].value_counts().to_dict(),
        "advanced_scenarios_count": int(len(advanced_scenarios)),
        "artifact_files": {k: str(v) for k, v in files.items()},
        "figures": figures,
        "notes": [
            "global_m2s é referência retrospectiva e não avança para NB11_FULL.",
            "train_m2s é o candidato principal causal quando atende ao suporte mínimo e à modelabilidade.",
            "train_p95 é sensibilidade causal; train_p99 é diagnóstico de severidade extrema.",
            "n_positivos_train e n_positivos_test são calculados pelo corte herdado do upstream.",
            "A Conclusão da Etapa é estática e deve ser preenchida após inspeção dos artefatos.",
        ],
    }
    files["summary"].write_text(json.dumps(json_ready(summary), indent=2, ensure_ascii=False), encoding="utf-8")

    log(f"Célula {cell_id} concluída: {cell_dir}")
    return {
        "cell_id": cell_id,
        "modelability_status": modelability_status,
        "stats": df_stats,
        "thresholds": df_thresholds,
        "episode_summary": df_episode_summary,
        "label_impact": df_label_impact,
        "decisions": df_decisions,
        "advanced_payload": advanced_payload,
        "summary": summary,
        "files": files,
    }


# ─────────────────────────────────────────────────────────────
# BLOCO 6 — Execução das 8 células e consolidação aggregate
# ─────────────────────────────────────────────────────────────

modelability_gate_df = load_modelability_gate()
results = []
for cell_id in ACTIVE_CELLS:
    result = process_cell(cell_id, modelability_gate_df)
    results.append(result)

# Consolidação tabular: as células continuam sendo réplicas independentes.
df_stats_all = pd.concat([r["stats"] for r in results], ignore_index=True)
df_thresholds_all = pd.concat([r["thresholds"] for r in results], ignore_index=True)
df_episode_summary_all = pd.concat([r["episode_summary"] for r in results], ignore_index=True)
df_label_impact_all = pd.concat([r["label_impact"] for r in results], ignore_index=True)
df_decisions_all = pd.concat([r["decisions"] for r in results], ignore_index=True)

aggregate_files = {
    "fail_rate_descriptive_stats_by_cell": AGGREGATE_DIR / "10_FULL_fail_rate_descriptive_stats_by_cell.csv",
    "thresholds_summary_by_cell": AGGREGATE_DIR / "10_FULL_thresholds_summary_by_cell.csv",
    "episode_summary_by_cell": AGGREGATE_DIR / "10_FULL_episode_summary_by_cell.csv",
    "label_impact_by_cell": AGGREGATE_DIR / "10_FULL_label_impact_by_cell.csv",
    "scenario_decisions_by_cell": AGGREGATE_DIR / "10_FULL_scenario_decisions_by_cell.csv",
    "advanced_to_nb11_by_cell": AGGREGATE_DIR / "10_FULL_advanced_to_nb11_by_cell.json",
    "diagnostics_summary": AGGREGATE_DIR / "10_FULL_diagnostics_summary.json",
    "delta_vs_canonical": AGGREGATE_DIR / "10_FULL_delta_vs_canonical.csv",
    "diff_summary": AGGREGATE_DIR / "10_FULL_diff_summary.csv",
    "diff_vs_canonical": AGGREGATE_DIR / "10_FULL_diff_vs_canonical.patch",
    "nbdiff_vs_canonical": AGGREGATE_DIR / "10_FULL_nbdiff_vs_canonical.txt",
    "artifact_manifest_sha256": AGGREGATE_DIR / "10_FULL_artifact_manifest_sha256.csv",
}

df_stats_all.to_csv(aggregate_files["fail_rate_descriptive_stats_by_cell"], index=False, encoding="utf-8")
df_thresholds_all.to_csv(aggregate_files["thresholds_summary_by_cell"], index=False, encoding="utf-8")
df_episode_summary_all.to_csv(aggregate_files["episode_summary_by_cell"], index=False, encoding="utf-8")
df_label_impact_all.to_csv(aggregate_files["label_impact_by_cell"], index=False, encoding="utf-8")
df_decisions_all.to_csv(aggregate_files["scenario_decisions_by_cell"], index=False, encoding="utf-8")

advanced_by_cell = {
    "notebook": "NB10_FULL_threshold_diagnostics",
    "active_cells": ACTIVE_CELLS,
    "advanced_to_nb11_by_cell": {r["cell_id"]: r["advanced_payload"]["advanced_to_nb11"] for r in results},
    "selection_rules": {
        "split_policy": "inherited_from_NB04_NB06_train_cutoff_idx_never_rederived_by_fraction",
        "main_threshold": "train_m2s",
        "historical_reference": "global_m2s_not_advanced",
        "persistence_focus": [1, 2],
    },
}
aggregate_files["advanced_to_nb11_by_cell"].write_text(
    json.dumps(json_ready(advanced_by_cell), indent=2, ensure_ascii=False),
    encoding="utf-8",
)

aggregate_summary = {
    "notebook": "NB10_FULL_threshold_diagnostics",
    "active_cells": ACTIVE_CELLS,
    "stage10_dir": str(STAGE10_DIR),
    "official_upstream_dirs": {
        "NB04_FULL": str(STAGE04_DIR),
        "NB06_FULL": str(STAGE06_DIR),
    },
    "legacy_dirs_ignored": [str(LEGACY_STAGE06_DIR)] if LEGACY_STAGE06_DIR.exists() else [],
    "parameters": {
        "train_fraction_reference_only": TRAIN_FRACTION_REFERENCE,
        "split_policy": "inherited_from_NB04_NB06_train_cutoff_idx_never_rederived_by_fraction",
        "std_ddof": STD_DDOF,
        "persistence_values": PERSISTENCE_VALUES,
        "threshold_methods": THRESHOLD_METHODS,
        "temporal_scenarios": TEMPORAL_SCENARIOS,
    },
    "cell_summaries": [r["summary"] for r in results],
    "decision_counts_global": df_decisions_all["decision_category"].value_counts().to_dict(),
    "advanced_scenarios_total": int(df_decisions_all["advance_to_nb11"].sum()),
    "artifact_files": {k: str(v) for k, v in aggregate_files.items()},
    "notes": [
        "NB07_FULL, NB08_FULL e NB09_FULL não existem neste ramo.",
        "NB10_FULL não treina modelos; a modelagem nasce no NB11_FULL.",
        "O corte treino/teste foi herdado dos artefatos NB04/NB06 por célula.",
        "O diretório oficial NB06 usado é 06_FULL_labeling_states.",
    ],
}
aggregate_files["diagnostics_summary"].write_text(
    json.dumps(json_ready(aggregate_summary), indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Delta mínimo em schema exigido pela estratégia.
delta_rows = [
    {
        "notebook_full": "10_threshold_diagnostics_FULL.ipynb",
        "notebook_canonical": "10_threshold_diagnostics.ipynb",
        "change_type": "I/O_PATH",
        "description": "Entradas e saídas movidas para 04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics; nenhuma escrita em 03-features.",
        "justified": True,
    },
    {
        "notebook_full": "10_threshold_diagnostics_FULL.ipynb",
        "notebook_canonical": "10_threshold_diagnostics.ipynb",
        "change_type": "CELL_LOOP",
        "description": "Processamento parametrizado por cell_id em ACTIVE_CELLS=list('abcdefgh'), com células tratadas como réplicas independentes.",
        "justified": True,
    },
    {
        "notebook_full": "10_threshold_diagnostics_FULL.ipynb",
        "notebook_canonical": "10_threshold_diagnostics.ipynb",
        "change_type": "FULL_DATASET_INPUT",
        "description": "Uso de artefatos NB04_FULL/NB06_FULL derivados do Parquet model-facing FULL, em vez dos artefatos canônicos Kaggle.",
        "justified": True,
    },
    {
        "notebook_full": "10_threshold_diagnostics_FULL.ipynb",
        "notebook_canonical": "10_threshold_diagnostics.ipynb",
        "change_type": "TRAIN_ONLY_THRESHOLD_BY_CELL",
        "description": "Limiar causal train-only estimado por célula, usando corte treino/teste herdado do NB04/NB06.",
        "justified": True,
    },
    {
        "notebook_full": "10_threshold_diagnostics_FULL.ipynb",
        "notebook_canonical": "10_threshold_diagnostics.ipynb",
        "change_type": "OUTPUT_PREFIX",
        "description": "Todos os artefatos novos recebem prefixo 10_FULL_ e são gravados em cell_<id>/aggregate.",
        "justified": True,
    },
    {
        "notebook_full": "10_threshold_diagnostics_FULL.ipynb",
        "notebook_canonical": "10_threshold_diagnostics.ipynb",
        "change_type": "AGGREGATION_BY_CELL",
        "description": "Tabelas aggregate consolidam resultados por célula, preservando cell_id e sem concatenar as séries como uma única série temporal.",
        "justified": True,
    },
    {
        "notebook_full": "10_threshold_diagnostics_FULL.ipynb",
        "notebook_canonical": "10_threshold_diagnostics.ipynb",
        "change_type": "REPORTING_ONLY",
        "description": "Inclusão de validações de diretório NB06 oficial, gate de modelabilidade e campos n_positivos_train/test para rastreabilidade.",
        "justified": True,
    },
]
pd.DataFrame(delta_rows).to_csv(aggregate_files["delta_vs_canonical"], index=False, encoding="utf-8")


def notebook_code_text(path: Path) -> str:
    nb = json.loads(path.read_text(encoding="utf-8"))
    parts = []
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") == "code":
            src = "".join(cell.get("source", []))
            parts.append(f"# %% cell {i}\n{src}\n")
    return "\n".join(parts)


def generate_diff_artifacts() -> None:
    canonical = CANONICAL_NOTEBOOK
    full_candidates = [
        FULL_NOTEBOOK,
        Path.cwd() / "10_threshold_diagnostics_FULL.ipynb",
    ]

    full = next((p for p in full_candidates if p.exists()), None)
    diff_rows = []

    if canonical.exists() and full is not None:
        canonical_text = notebook_code_text(canonical).splitlines(keepends=True)
        full_text = notebook_code_text(full).splitlines(keepends=True)

        diff = list(difflib.unified_diff(
            canonical_text,
            full_text,
            fromfile=str(canonical),
            tofile=str(full),
        ))

        aggregate_files["diff_vs_canonical"].write_text("".join(diff), encoding="utf-8")
        aggregate_files["nbdiff_vs_canonical"].write_text("".join(diff), encoding="utf-8")

        diff_rows.append({
            "artifact": "diff_vs_canonical",
            "status": "generated",
            "canonical_found": True,
            "full_found": True,
            "canonical_path": str(canonical),
            "full_path": str(full),
            "diff_lines": len(diff),
            "note": "Diff textual de células de código gerado por difflib.",
        })
    else:
        note = (
            "Não foi possível gerar diff. "
            f"canonical_exists={canonical.exists()} canonical={canonical}; "
            f"full_found={full is not None}; "
            f"full_candidates={[str(p) for p in full_candidates]}"
        )

        aggregate_files["diff_vs_canonical"].write_text(note + "\n", encoding="utf-8")
        aggregate_files["nbdiff_vs_canonical"].write_text(note + "\n", encoding="utf-8")

        diff_rows.append({
            "artifact": "diff_vs_canonical",
            "status": "not_generated",
            "canonical_found": canonical.exists(),
            "full_found": full is not None,
            "canonical_path": str(canonical),
            "full_path": str(full) if full is not None else "",
            "diff_lines": 0,
            "note": note,
        })

    pd.DataFrame(diff_rows).to_csv(
        aggregate_files["diff_summary"],
        index=False,
        encoding="utf-8",
    )


generate_diff_artifacts()
manifest_df = build_artifact_manifest(STAGE10_DIR, aggregate_files["artifact_manifest_sha256"])

print("\n=== NB10_FULL — arquivos aggregate gerados ===")
display(pd.DataFrame({
    "artifact": list(aggregate_files.keys()),
    "path": [str(p) for p in aggregate_files.values()],
    "exists": [p.exists() for p in aggregate_files.values()],
}))

print("\n=== NB10_FULL — resumo por célula ===")
cell_overview = []
for r in results:
    cell_id = r["cell_id"]
    decisions = r["decisions"]
    cell_overview.append({
        "cell_id": cell_id,
        "modelability_status": r["modelability_status"],
        "advanced_to_nb11_count": int(decisions["advance_to_nb11"].sum()),
        "historical_reference_count": int((decisions["decision_category"] == "historical_reference").sum()),
        "sensitivity_only_count": int((decisions["decision_category"] == "sensitivity_only").sum()),
        "do_not_advance_count": int((decisions["decision_category"] == "do_not_advance").sum()),
    })
display(pd.DataFrame(cell_overview))

print("\n=== NB10_FULL — cenários avançados para NB11_FULL ===")
cols = [
    "cell_id", "scenario_id", "scenario_family", "window_minutes", "k_minutes", "h_minutes",
    "threshold_method", "persistence_min_windows", "episode_count", "supervised_rows",
    "positive_rate", "n_positivos_train", "n_positivos_test", "scientific_justification",
]
display(df_decisions_all.loc[df_decisions_all["advance_to_nb11"] == True, [c for c in cols if c in df_decisions_all.columns]])

print("\n=== NB10_FULL — manifesto SHA-256 ===")
display(manifest_df.head(20))

log("Execução concluída. A Conclusão da Etapa deve ser preenchida estaticamente após análise dos artefatos.")


[Bootstrap] Google Drive já montado.
[Bootstrap] Atualizando repositório (git pull)...
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
DRIVE_ROOT = /content/drive/MyDrive/Mestrado
FULL_REPORTS_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream
STAGE04_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes
STAGE06_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/06_FULL_labeling_states
STAGE10_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics
AGGREGATE_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics/aggregate
ACTIVE_CELLS = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
[NB10_FULL_threshold_diagnostics] Parâmetros principais definidos.
TRAIN_FRACTION_REFERENCE = 0.8 (somente assert; split herdado do NB04/NB06)
STD_DDOF = 0
PERSISTENCE_VALUES = [1, 2, 3]
THRESHOLD_METHODS = ['global_m2s', 'train_m2s', 'train_p95', 'train

,artifact,path,exists
0,fail_rate_descriptive_stats_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
1,thresholds_summary_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
2,episode_summary_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
3,label_impact_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
4,scenario_decisions_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
5,advanced_to_nb11_by_cell,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
6,diagnostics_summary,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
7,delta_vs_canonical,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
8,diff_summary,/content/drive/MyDrive/Mestrado/04-reports/99_...,True
9,diff_vs_canonical,/content/drive/MyDrive/Mestrado/04-reports/99_...,True



=== NB10_FULL — resumo por célula ===


,cell_id,modelability_status,advanced_to_nb11_count,historical_reference_count,sensitivity_only_count,do_not_advance_count
0,a,has_episode_support_pending_nb06_labels,9,18,33,12
1,b,has_episode_support_pending_nb06_labels,12,18,33,9
2,c,has_episode_support_pending_nb06_labels,9,18,33,12
3,d,has_episode_support_pending_nb06_labels,2,18,26,26
4,e,has_episode_support_pending_nb06_labels,6,18,33,15
5,f,has_episode_support_pending_nb06_labels,6,18,36,12
6,g,has_episode_support_pending_nb06_labels,12,18,39,3
7,h,has_episode_support_pending_nb06_labels,6,18,33,15



=== NB10_FULL — cenários avançados para NB11_FULL ===


,cell_id,scenario_id,scenario_family,window_minutes,k_minutes,h_minutes,threshold_method,persistence_min_windows,episode_count,supervised_rows,positive_rate,n_positivos_train,n_positivos_test,scientific_justification
3,a,cell_a_W5_K24_H12_train_m2s_P1,W5_K24_H12,5,120,60,train_m2s,1,191,7363,0.140432,890,144,Cenário causal train_m2s com persistência foco...
4,a,cell_a_W5_K24_H12_train_m2s_P2,W5_K24_H12,5,120,60,train_m2s,2,84,8089,0.062801,446,62,Cenário causal train_m2s com persistência foco...
15,a,cell_a_W5_K12_H6_train_m2s_P1,W5_K12_H6,5,60,30,train_m2s,1,191,7845,0.082983,573,78,Cenário causal train_m2s com persistência foco...
16,a,cell_a_W5_K12_H6_train_m2s_P2,W5_K12_H6,5,60,30,train_m2s,2,84,8293,0.038225,285,32,Cenário causal train_m2s com persistência foco...
27,a,cell_a_W5_K24_H6_train_m2s_P1,W5_K24_H6,5,120,30,train_m2s,1,191,7369,0.088343,573,78,Cenário causal train_m2s com persistência foco...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519,h,cell_h_W5_K12_H6_train_m2s_P1,W5_K12_H6,5,60,30,train_m2s,1,225,7220,0.166066,979,220,Cenário causal train_m2s com persistência foco...
531,h,cell_h_W5_K24_H6_train_m2s_P1,W5_K24_H6,5,120,30,train_m2s,1,225,6703,0.178875,979,220,Cenário causal train_m2s com persistência foco...
543,h,cell_h_W10_K12_H6_train_m2s_P1,W10_K12_H6,10,120,60,train_m2s,1,131,3565,0.187658,546,123,Cenário causal train_m2s com persistência foco...
555,h,cell_h_W10_K6_H3_train_m2s_P1,W10_K6_H3,10,60,30,train_m2s,1,131,3807,0.096139,297,69,Cenário causal train_m2s com persistência foco...



=== NB10_FULL — manifesto SHA-256 ===


,relative_path,size_bytes,sha256
0,aggregate/10_FULL_advanced_to_nb11_by_cell.json,110591,fabaea29bd35817c0da0f4874ffe69d7a8d78a92f55544...
1,aggregate/10_FULL_delta_vs_canonical.csv,1515,39cac47c56c8cc90006d64a4da67e14805ecc849c00e5a...
2,aggregate/10_FULL_diagnostics_summary.json,79015,49226e51e2c8201567b312a10c1a17ec530ae06b08f5cb...
3,aggregate/10_FULL_diff_summary.csv,328,b10d03ce58bbef518cc6a0c2fa8762de41d7fb9b5dae3c...
4,aggregate/10_FULL_diff_vs_canonical.patch,114034,cb1aec548ea9f2ef5a226c281a94da53f289f7a117a05b...
5,aggregate/10_FULL_episode_summary_by_cell.csv,214964,259c9b478a0bb916dbc2e522b15f14ee5e8785d0451386...
6,aggregate/10_FULL_fail_rate_descriptive_stats_...,13649,c72ae709eda9a499ce9ec5952d005a3cc10b06c8c9dff3...
7,aggregate/10_FULL_label_impact_by_cell.csv,208199,120c3cb7376eb4d6e6d3465cb45ebe7e48e5a9b61472b2...
8,aggregate/10_FULL_nbdiff_vs_canonical.txt,114034,cb1aec548ea9f2ef5a226c281a94da53f289f7a117a05b...
9,aggregate/10_FULL_scenario_decisions_by_cell.csv,326154,6da09ee5fdc49792d3e919bfef781a899e97d650913922...


[NB10_FULL_threshold_diagnostics] Execução concluída. A Conclusão da Etapa deve ser preenchida estaticamente após análise dos artefatos.


## 10. Conclusão da Etapa — NB10_FULL

### 10.1 Execução e escopo

O NB10_FULL foi executado para as oito células Borg (`a`–`h`) no diretório oficial do downstream:

```text
/content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/10_FULL_threshold_diagnostics
```

A etapa cumpriu o papel metodológico previsto para o ramo `_FULL`: avaliar cenários de limiar, granularidade temporal, parâmetros K/H e persistência mínima por célula, sem criar nem depender de NB07_FULL, NB08_FULL ou NB09_FULL. A modelagem propriamente dita permanece reservada ao NB11_FULL, e a consolidação final ao bloco NB14_FULL–NB16_FULL.

A execução manteve o princípio de replicação ampliada de robustez/escala, não de substituição do canônico. O pipeline canônico continua sendo a base principal da dissertação; o `_FULL` fornece evidência adicional sobre a robustez da formulação causal/supervisionada no Borg 2019 completo, com segmentação por célula.

### 10.2 Herança do corte treino/teste

O ponto metodológico mais crítico foi validado: o corte treino/teste não foi rederivado por fração local. Em todas as células, o NB10_FULL herdou `train_cutoff_idx` e `train_cutoff_bucket_id` diretamente da série scored do NB04_FULL (`NB04_series_columns`).

| célula   | fonte do corte      |   train_cutoff_idx |   train_cutoff_bucket_id |   w5_rows |   w5_train_ratio |   w10_train_cutoff_idx |   w10_train_ratio |
|:---------|:--------------------|-------------------:|-------------------------:|----------:|-----------------:|-----------------------:|------------------:|
| a        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |
| b        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |
| c        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |
| d        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |
| e        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |
| f        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |
| g        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |
| h        | NB04_series_columns |               7142 |                     7143 |      8928 |         0.799955 |                   3571 |          0.799955 |

O corte herdado preserva a razão temporal próxima de 80/20 sem usar `floor(len × fração)`. Isso evita a inconsistência de fronteira já identificada anteriormente e garante que os limiares `train_m2s`, `train_p95` e `train_p99` sejam calculados sobre a mesma região de treino comprometida pelo NB04_FULL/NB06_FULL.

### 10.3 Cenários avaliados

Foram avaliados 72 cenários por célula, combinando:

- 2 granularidades: 5 e 10 minutos;
- 6 famílias temporais W/K/H;
- 4 métodos de limiar: `global_m2s`, `train_m2s`, `train_p95`, `train_p99`;
- 3 níveis de persistência mínima: P1, P2 e P3.

No total, o NB10_FULL avaliou 576 combinações célula-cenário. A distribuição das decisões foi:

| categoria            |   cenários |   percentual |
|:---------------------|-----------:|-------------:|
| sensitivity_only     |        266 |         46.2 |
| historical_reference |        144 |         25.0 |
| do_not_advance       |        104 |         18.1 |
| advance_to_nb11      |         62 |         10.8 |

Por célula, a distribuição foi:

| célula   |   advance_to_nb11 |   sensitivity_only |   historical_reference |   do_not_advance |
|:---------|------------------:|-------------------:|-----------------------:|-----------------:|
| a        |                 9 |                 33 |                     18 |               12 |
| b        |                12 |                 33 |                     18 |                9 |
| c        |                 9 |                 33 |                     18 |               12 |
| d        |                 2 |                 26 |                     18 |               26 |
| e        |                 6 |                 33 |                     18 |               15 |
| f        |                 6 |                 36 |                     18 |               12 |
| g        |                12 |                 39 |                     18 |                3 |
| h        |                 6 |                 33 |                     18 |               15 |

A decisão está cientificamente coerente: os cenários `global_m2s` foram preservados como referência histórica, mas não avançam no ramo causal `_FULL`; os cenários `train_m2s` concentram os avanços ao NB11_FULL; `train_p95` e `train_p99` permanecem principalmente como sensibilidade ou diagnóstico de severidade.

### 10.4 Cenário principal de continuidade com o canônico

O cenário de maior continuidade metodológica com a formulação canônica da dissertação é:

```text
W5_K24_H12_train_m2s_P1
```

Ele preserva janela de 5 minutos, K=24, H=12, persistência mínima P=1 e limiar causal `train_m2s`, agora estimado por célula e com corte herdado do NB04_FULL. Os resultados por célula foram:

| célula   |   threshold_train_m2s |   janelas críticas |   episódios |   amostras |   positivos |   taxa positiva (%) |   positivos treino |   positivos teste | avança NB11   |
|:---------|----------------------:|-------------------:|------------:|-----------:|------------:|--------------------:|-------------------:|------------------:|:--------------|
| a        |                0.0290 |                441 |         191 |       7363 |        1034 |             14.0432 |                890 |               144 | sim           |
| b        |                0.0315 |                478 |         219 |       7710 |        1343 |             17.4189 |               1196 |               147 | sim           |
| c        |                0.0186 |                380 |         178 |       7261 |        1241 |             17.0913 |               1067 |               174 | sim           |
| d        |                0.0375 |                413 |         130 |       8291 |         507 |              6.1151 |                494 |                13 | sim           |
| e        |                0.0108 |                245 |         110 |       7740 |         848 |             10.9561 |                709 |               139 | sim           |
| f        |                0.0125 |                247 |         135 |       7172 |        1280 |             17.8472 |               1049 |               231 | sim           |
| g        |                0.0148 |                602 |         295 |       7154 |        2141 |             29.9273 |               1651 |               490 | sim           |
| h        |                0.0140 |                293 |         225 |       6697 |        2113 |             31.5514 |               1729 |               384 | sim           |

Os totais de janelas críticas e episódios coincidem com os resultados esperados do NB04_FULL para o cenário `TRAIN_M2S` por célula: `a`=441/191, `b`=478/219, `c`=380/178, `d`=413/130, `e`=245/110, `f`=247/135, `g`=602/295 e `h`=293/225. Essa consistência confirma que o NB10_FULL não redefiniu a criticidade principal; apenas ampliou o diagnóstico de cenários.

A célula `d` merece atenção especial: embora avance no cenário principal, possui apenas 13 positivos no teste nesse cenário. Portanto, deve ser tratada no NB11_FULL como célula modelável com suporte restrito, exigindo cautela na interpretação das métricas de teste.

### 10.5 Artefatos gerados

A execução produziu os artefatos esperados por célula e em `aggregate`. Foram gerados CSVs, JSONs, Parquets de episódios/estados e figuras diagnósticas para todas as células.

Principais artefatos agregados:

```text
10_FULL_thresholds_summary_by_cell.csv
10_FULL_fail_rate_descriptive_stats_by_cell.csv
10_FULL_episode_summary_by_cell.csv
10_FULL_label_impact_by_cell.csv
10_FULL_scenario_decisions_by_cell.csv
10_FULL_advanced_to_nb11_by_cell.json
10_FULL_diagnostics_summary.json
10_FULL_delta_vs_canonical.csv
10_FULL_artifact_manifest_sha256.csv
```

O manifesto SHA-256 contém 195 artefatos rastreados. Todos os arquivos listados no manifesto foram encontrados, os tamanhos conferem e os hashes SHA-256 foram validados. O arquivo do próprio manifesto não é listado dentro dele, o que explica a diferença entre 196 arquivos físicos e 195 entradas no manifesto.

### 10.6 Validações de aderência

A versão executada do notebook não contém mais a lógica reprovada de corte 70/30 nem rederivação por `np.floor`:

- `TRAIN_FRACTION = 0.70`: 0 ocorrência(s)
- `TRAIN_FRACTION=0.70`: 0 ocorrência(s)
- `np.floor`: 0 ocorrência(s)
- `floor(len`: 0 ocorrência(s)
- `n_train = int`: 0 ocorrência(s)

O SHA-256 do notebook executado é:

```text
f8e23575c291090aeee0dab2d37e4e157b01580cee1f7367561e3a394fac2d11
```

O arquivo `10_FULL_delta_vs_canonical.csv` foi gerado com o schema exigido pela estratégia do `_FULL`:

```text
notebook_full, notebook_canonical, change_type, description, justified
```

As alterações registradas são compatíveis com a política de delta mínimo: caminhos de entrada/saída, execução por célula, entrada `_FULL`, herança do corte treino/teste, cálculo de limiar por célula, prefixos `_FULL`, agregação por célula e relatório.

### 10.7 Pendência de rastreabilidade formal

A única pendência observada é de rastreabilidade formal, não de resultado científico: o diff canônico↔FULL não foi gerado porque o notebook FULL não estava localizado em `REPO_DIR/notebooks` durante a execução.

Status registrado em `10_FULL_diff_summary.csv`:

```text
not_generated
```

Observação registrada:

```text
Não foi possível gerar diff porque o notebook FULL não foi localizado em REPO_DIR/notebooks. Copie-o para o repositório antes da execução para obter diff real.
```

Essa pendência não invalida os artefatos científicos do NB10_FULL, mas deve ser resolvida antes do fechamento documental da etapa: copiar o notebook FULL para o diretório `notebooks/` do repositório, reexecutar a célula de auditoria ou gerar manualmente o diff canônico↔FULL e substituir os placeholders `10_FULL_diff_vs_canonical.patch` e `10_FULL_nbdiff_vs_canonical.txt`.

### 10.8 Decisão para o próximo notebook

O NB10_FULL está aprovado como etapa científica e operacional do ramo `_FULL`. Ele confirma que a formulação causal `train_m2s` permanece bem definida nas oito células, embora com heterogeneidade importante entre elas.

O NB11_FULL deve consumir preferencialmente:

```text
10_FULL_advanced_to_nb11_by_cell.json
10_FULL_scenario_decisions_by_cell.csv
10_FULL_label_impact_by_cell.csv
```

Como cenário principal de continuidade, recomenda-se iniciar o NB11_FULL por:

```text
W5_K24_H12_train_m2s_P1
```

Em seguida, os demais cenários `advance_to_nb11 = True` podem ser tratados como expansão controlada ou sensibilidade tabular, sempre preservando a segmentação por célula e sem concatenar as oito células como uma única série temporal.

### 10.9 Veredito final

A execução do NB10_FULL está correta quanto à lógica científica, diretórios, herança do corte treino/teste, geração de cenários, consolidação por célula, prefixos `_FULL` e manifesto. A etapa pode avançar para o NB11_FULL.

A ressalva é exclusivamente documental: gerar o diff real canônico↔FULL antes de considerar a rastreabilidade da etapa completamente fechada.
